In [2]:
# 01_ccm_prep.ipynb

In [3]:
# Imports and paths

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("../..")   # change if needed
PROCESSED = BASE / "data" / "processed" / "CY-Bench"
PROCESSED.mkdir(parents=True, exist_ok=True)

CROP = "wheat"
COUNTRY = "IN"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [4]:
# Load panel from preprocessing notebook

panel_path = PROCESSED / "wheat_IN_panel_full_year.csv"
panel = pd.read_csv(panel_path)

panel.shape

(7552, 45)

In [5]:
# Basic schema check

panel.columns.tolist()

['crop_name',
 'country_code',
 'adm_id',
 'harvest_year',
 'yield',
 'production',
 'harvest_area',
 'sos',
 'eos',
 'crop_area',
 'crop_area_percentage',
 'latitude',
 'longitude',
 'region_area',
 'awc',
 'bulk_density',
 'drainage_class',
 'avg_tmin',
 'avg_tmax',
 'avg_tavg',
 'avg_prec',
 'sum_prec',
 'avg_rad',
 'avg_et0',
 'avg_vpd',
 'avg_cwb',
 'n_meteo_obs',
 'avg_ssm',
 'min_ssm',
 'max_ssm',
 'avg_rsm',
 'min_rsm',
 'max_rsm',
 'n_sm_obs',
 'avg_ndvi',
 'min_ndvi',
 'max_ndvi',
 'std_ndvi',
 'n_ndvi_obs',
 'avg_fpar',
 'min_fpar',
 'max_fpar',
 'std_fpar',
 'n_fpar_obs',
 'n_available_blocks']

In [6]:
# Standardize key columns and types

panel = panel.copy()

if "crop_name" in panel.columns:
    panel["crop_name"] = panel["crop_name"].astype(str).str.strip().str.lower()

if "country_code" in panel.columns:
    panel["country_code"] = panel["country_code"].astype(str).str.strip().str.upper()

if "adm_id" in panel.columns:
    panel["adm_id"] = panel["adm_id"].astype(str).str.strip()

panel["harvest_year"] = pd.to_numeric(panel["harvest_year"], errors="coerce").astype("Int64")

panel = panel.sort_values(["adm_id", "harvest_year"]).reset_index(drop=True)

panel.head()

,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,crop_area,crop_area_percentage,latitude,longitude,region_area,awc,bulk_density,drainage_class,avg_tmin,avg_tmax,avg_tavg,avg_prec,sum_prec,avg_rad,avg_et0,avg_vpd,avg_cwb,n_meteo_obs,avg_ssm,min_ssm,max_ssm,avg_rsm,min_rsm,max_rsm,n_sm_obs,avg_ndvi,min_ndvi,max_ndvi,std_ndvi,n_ndvi_obs,avg_fpar,min_fpar,max_fpar,std_fpar,n_fpar_obs,n_available_blocks
0,wheat,IN,IN-01-0047,2017,1.000,10.0,10.0,320.213,92.669,20.2,0.243,17.52128,81.18046,8316.0,14.203,1.506,4.0,23.450277,33.001384,27.813666,3.662170,1336.692,1.838194e+07,4.500485,26.257896,-0.838315,365.0,4.605701,2.041,8.040,256.749784,235.919,319.130,365.0,0.587410,0.374,0.758,0.128424,39.0,44.094639,24.974,62.886,13.939727,36.0,4
1,wheat,IN,IN-01-0049,2010,1.500,60.0,40.0,284.497,87.670,72.4,0.635,16.39034,79.72043,11398.0,15.192,1.542,4.0,23.812932,32.932452,27.655723,3.923219,1431.975,1.803340e+07,4.806238,26.328907,-0.883019,365.0,4.711455,1.947,8.433,259.160753,233.007,317.711,365.0,0.546682,0.284,0.761,0.137065,44.0,47.217833,27.859,71.492,14.510811,36.0,4
2,wheat,IN,IN-01-0051,2001,0.698,880.0,1260.0,313.989,102.880,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,22.820247,32.929427,27.616770,1.545647,564.161,1.922248e+07,5.392036,30.792145,-3.846389,365.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.364452,0.190,0.615,0.108494,42.0,28.438778,12.659,50.322,12.020800,36.0,3
3,wheat,IN,IN-01-0051,2002,1.000,1000.0,1000.0,313.989,102.880,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,22.865921,33.240953,27.804756,1.303027,475.605,1.973802e+07,5.441342,31.526392,-4.138315,365.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.386744,0.247,0.522,0.088545,43.0,31.348917,18.032,50.017,11.102711,36.0,3
4,wheat,IN,IN-01-0051,2003,0.304,210.0,690.0,313.989,102.880,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,23.324778,33.771510,28.309710,1.217356,444.335,1.922673e+07,5.597868,33.526879,-4.380512,365.0,3.771593,0.827,7.338,277.465955,270.581,312.943,334.0,0.366091,0.193,0.595,0.121144,44.0,28.397778,13.847,53.197,13.007601,36.0,4


In [7]:
# Decide dynamic vs static variables

id_cols = ["crop_name", "country_code", "adm_id", "harvest_year"]

dynamic_candidates = [
    "yield",
    "production",
    "harvest_area",
    "avg_ssm",
    "avg_rsm",
    "avg_ndvi",
    "avg_fpar",
    "avg_tmin",
    "avg_tmax",
    "avg_tavg",
    "avg_prec",
    "sum_prec",
    "avg_rad",
    "avg_et0",
    "avg_vpd",
    "avg_cwb",
]

static_candidates = [
    "sos",
    "eos",
    "crop_area",
    "crop_area_percentage",
    "latitude",
    "longitude",
    "region_area",
    "awc",
    "bulk_density",
    "drainage_class",
]

dynamic_cols = [c for c in dynamic_candidates if c in panel.columns]
static_cols = [c for c in static_candidates if c in panel.columns]

print("dynamic_cols:", dynamic_cols)
print("static_cols:", static_cols)

dynamic_cols: ['yield', 'production', 'harvest_area', 'avg_ssm', 'avg_rsm', 'avg_ndvi', 'avg_fpar', 'avg_tmin', 'avg_tmax', 'avg_tavg', 'avg_prec', 'sum_prec', 'avg_rad', 'avg_et0', 'avg_vpd', 'avg_cwb']
static_cols: ['sos', 'eos', 'crop_area', 'crop_area_percentage', 'latitude', 'longitude', 'region_area', 'awc', 'bulk_density', 'drainage_class']


In [8]:
# Keep only relevant columns for CCM prep

keep_cols = id_cols + dynamic_cols + static_cols
panel = panel[keep_cols].copy()

panel.shape

(7552, 30)

In [9]:
# Missingness summary for dynamic variables

dynamic_missing = (
    panel[dynamic_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "variable"})
)

dynamic_missing

,variable,missing_fraction
0,avg_ssm,0.110832
1,avg_rsm,0.110832
2,production,0.000000
3,yield,0.000000
4,harvest_area,0.000000
5,avg_ndvi,0.000000
6,avg_fpar,0.000000
7,avg_tmin,0.000000
8,avg_tmax,0.000000
9,avg_tavg,0.000000


In [10]:
# Count yearly observations per adm_id

unit_counts = (
    panel.groupby("adm_id", as_index=False)
    .agg(
        n_years=("harvest_year", "count"),
        year_min=("harvest_year", "min"),
        year_max=("harvest_year", "max"),
    )
    .sort_values(["n_years", "adm_id"], ascending=[False, True])
    .reset_index(drop=True)
)

unit_counts.head(20)

,adm_id,n_years,year_min,year_max
0,IN-01-0051,17,2001,2017
1,IN-02-0902,17,2001,2017
2,IN-02-0903,17,2001,2017
3,IN-02-0904,17,2001,2017
4,IN-02-0905,17,2001,2017
5,IN-02-0906,17,2001,2017
6,IN-02-0907,17,2001,2017
7,IN-02-0908,17,2001,2017
8,IN-02-0909,17,2001,2017
9,IN-02-0912,17,2001,2017


In [11]:
# Filter to units with enough years for CCM

MIN_SERIES_LEN = 10

eligible_units = unit_counts.loc[unit_counts["n_years"] >= MIN_SERIES_LEN, "adm_id"].tolist()

panel_eligible = (
    panel[panel["adm_id"].isin(eligible_units)]
    .sort_values(["adm_id", "harvest_year"])
    .reset_index(drop=True)
)

print("eligible units:", len(eligible_units))
print("panel_eligible shape:", panel_eligible.shape)

eligible units: 441
panel_eligible shape: (7319, 30)


In [12]:
# Helper to z-score within each adm_id for dynamic variables only

def zscore_within_unit(df, group_col, cols):
    df = df.copy()
    for col in cols:
        mu = df.groupby(group_col)[col].transform("mean")
        sd = df.groupby(group_col)[col].transform("std")
        df[col + "_z"] = (df[col] - mu) / sd.replace(0, np.nan)
    return df

In [13]:
# Apply within-unit standardization

panel_z = zscore_within_unit(panel_eligible, "adm_id", dynamic_cols)

z_cols = [c + "_z" for c in dynamic_cols]
panel_z.head()

,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,avg_ssm,avg_rsm,avg_ndvi,avg_fpar,avg_tmin,avg_tmax,avg_tavg,avg_prec,sum_prec,avg_rad,avg_et0,avg_vpd,avg_cwb,sos,eos,crop_area,crop_area_percentage,latitude,longitude,region_area,awc,bulk_density,drainage_class,yield_z,production_z,harvest_area_z,avg_ssm_z,avg_rsm_z,avg_ndvi_z,avg_fpar_z,avg_tmin_z,avg_tmax_z,avg_tavg_z,avg_prec_z,sum_prec_z,avg_rad_z,avg_et0_z,avg_vpd_z,avg_cwb_z
0,wheat,IN,IN-01-0051,2001,0.698,880.0,1260.0,NaN,NaN,0.364452,28.438778,22.820247,32.929427,27.616770,1.545647,564.161,1.922248e+07,5.392036,30.792145,-3.846389,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-0.232675,1.843314,2.216593,NaN,NaN,-1.429948,-1.704969,-0.510498,-0.958153,-0.536452,-0.786422,-0.789498,0.163328,0.254394,-0.799075,-0.691319
1,wheat,IN,IN-01-0051,2002,1.000,1000.0,1000.0,NaN,NaN,0.386744,31.348917,22.865921,33.240953,27.804756,1.303027,475.605,1.973802e+07,5.441342,31.526392,-4.138315,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,0.329997,2.242928,1.475493,NaN,NaN,-0.398883,-0.245613,-0.365002,-0.177151,-0.001281,-1.361055,-1.364387,1.887200,0.627640,-0.224358,-1.243257
2,wheat,IN,IN-01-0051,2003,0.304,210.0,690.0,3.771593,277.465955,0.366091,28.397778,23.324778,33.771510,28.309710,1.217356,444.335,1.922673e+07,5.597868,33.526879,-4.380512,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-0.966757,-0.387860,0.591874,-1.610043,-1.342326,-1.354161,-1.725530,1.096705,1.152964,1.436250,-1.563964,-1.567386,0.177544,1.812522,1.341483,-1.701174
3,wheat,IN,IN-01-0051,2004,0.231,90.0,390.0,3.742656,276.584082,0.367689,29.201611,22.554311,32.927232,27.427087,1.606833,588.101,1.921154e+07,5.342434,30.978757,-3.735601,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-1.102767,-0.787473,-0.263241,-1.677796,-1.518685,-1.280249,-1.322429,-1.357644,-0.963657,-1.076451,-0.641504,-0.634084,0.126741,-0.121080,-0.653009,-0.481854
4,wheat,IN,IN-01-0051,2005,0.592,420.0,710.0,4.347534,286.638466,0.380045,30.443778,22.837874,32.939112,27.566255,1.999808,729.930,1.917189e+07,5.207014,30.445112,-3.207205,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-0.430169,0.311463,0.648882,-0.261523,0.492009,-0.708720,-0.699516,-0.454345,-0.933873,-0.680261,0.289240,0.286642,-0.005818,-1.146197,-1.070708,0.517171


In [14]:
# Check that identifiers are preserved

required_id_cols = ["crop_name", "country_code", "adm_id", "harvest_year"]
missing_id_cols = [c for c in required_id_cols if c not in panel_z.columns]

if missing_id_cols:
    raise ValueError(f"Missing identifier columns: {missing_id_cols}")

panel_z[required_id_cols].head()

,crop_name,country_code,adm_id,harvest_year
0,wheat,IN,IN-01-0051,2001
1,wheat,IN,IN-01-0051,2002
2,wheat,IN,IN-01-0051,2003
3,wheat,IN,IN-01-0051,2004
4,wheat,IN,IN-01-0051,2005


In [15]:
# Choose first-pass CCM variable pairs

pair_specs = [
    ("avg_ssm_z", "avg_ndvi_z"),
    ("avg_ssm_z", "avg_fpar_z"),
    ("avg_ssm_z", "yield_z"),
    ("avg_rsm_z", "avg_ndvi_z"),
    ("avg_rsm_z", "avg_fpar_z"),
    ("avg_rsm_z", "yield_z"),
    ("avg_tavg_z", "avg_cwb_z"),
    ("avg_prec_z", "yield_z"),
]

pair_specs = [(x, y) for x, y in pair_specs if x in panel_z.columns and y in panel_z.columns]
pair_specs

[('avg_ssm_z', 'avg_ndvi_z'),
 ('avg_ssm_z', 'avg_fpar_z'),
 ('avg_ssm_z', 'yield_z'),
 ('avg_rsm_z', 'avg_ndvi_z'),
 ('avg_rsm_z', 'avg_fpar_z'),
 ('avg_rsm_z', 'yield_z'),
 ('avg_tavg_z', 'avg_cwb_z'),
 ('avg_prec_z', 'yield_z')]

In [16]:
# Helper to build a CCM-ready pair frame

def make_pair_frame(df, x_col, y_col, min_len=MIN_SERIES_LEN):
    keep_cols = ["crop_name", "country_code", "adm_id", "harvest_year"] + static_cols + [x_col, y_col]
    out = df[keep_cols].copy()
    out = out.dropna(subset=[x_col, y_col]).copy()
    out = out.sort_values(["adm_id", "harvest_year"]).reset_index(drop=True)

    lengths = (
        out.groupby("adm_id", as_index=False)
        .agg(n_obs=("harvest_year", "count"))
    )

    keep_units = lengths.loc[lengths["n_obs"] >= min_len, "adm_id"].tolist()
    out = out[out["adm_id"].isin(keep_units)].copy()
    out = out.sort_values(["adm_id", "harvest_year"]).reset_index(drop=True)

    return out

In [17]:
# Build pair tables

pair_frames = {}

for x_col, y_col in pair_specs:
    pair_name = f"{x_col.replace('_z','')}_{y_col.replace('_z','')}"
    pair_df = make_pair_frame(panel_z, x_col, y_col, min_len=MIN_SERIES_LEN)
    pair_frames[pair_name] = pair_df
    print(pair_name, pair_df.shape)

avg_ssm_avg_ndvi (6460, 16)
avg_ssm_avg_fpar (6460, 16)
avg_ssm_yield (6460, 16)
avg_rsm_avg_ndvi (6460, 16)
avg_rsm_avg_fpar (6460, 16)
avg_rsm_yield (6460, 16)
avg_tavg_avg_cwb (7319, 16)
avg_prec_yield (7319, 16)


In [18]:
# Diagnostics for pair tables

pair_diagnostics = []

for pair_name, df_pair in pair_frames.items():
    pair_diagnostics.append({
        "pair_name": pair_name,
        "n_rows": len(df_pair),
        "n_units": df_pair["adm_id"].nunique() if len(df_pair) else 0,
        "year_min": df_pair["harvest_year"].min() if len(df_pair) else np.nan,
        "year_max": df_pair["harvest_year"].max() if len(df_pair) else np.nan,
    })

pair_diagnostics = pd.DataFrame(pair_diagnostics).sort_values("pair_name").reset_index(drop=True)
pair_diagnostics

,pair_name,n_rows,n_units,year_min,year_max
0,avg_prec_yield,7319,441,2001,2017
1,avg_rsm_avg_fpar,6460,438,2003,2017
2,avg_rsm_avg_ndvi,6460,438,2003,2017
3,avg_rsm_yield,6460,438,2003,2017
4,avg_ssm_avg_fpar,6460,438,2003,2017
5,avg_ssm_avg_ndvi,6460,438,2003,2017
6,avg_ssm_yield,6460,438,2003,2017
7,avg_tavg_avg_cwb,7319,441,2001,2017


In [19]:
# Save CCM-ready panel and pair files

panel_z_path = PROCESSED / "wheat_IN_ccm_ready_panel.csv"
pair_diag_path = PROCESSED / "wheat_IN_ccm_pair_diagnostics.csv"
unit_counts_path = PROCESSED / "wheat_IN_ccm_unit_counts.csv"

panel_z.to_csv(panel_z_path, index=False)
pair_diagnostics.to_csv(pair_diag_path, index=False)
unit_counts.to_csv(unit_counts_path, index=False)

for pair_name, df_pair in pair_frames.items():
    out_path = PROCESSED / f"ccm_pair_{pair_name}.csv"
    df_pair.to_csv(out_path, index=False)

panel_z_path

PosixPath('../../data/processed/CY-Bench/wheat_IN_ccm_ready_panel.csv')

In [20]:
# Quick preview of one pair file

first_pair_name = sorted(pair_frames.keys())[0]
pair_frames[first_pair_name].head()

,crop_name,country_code,adm_id,harvest_year,sos,eos,crop_area,crop_area_percentage,latitude,longitude,region_area,awc,bulk_density,drainage_class,avg_prec_z,yield_z
0,wheat,IN,IN-01-0051,2001,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-0.786422,-0.232675
1,wheat,IN,IN-01-0051,2002,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-1.361055,0.329997
2,wheat,IN,IN-01-0051,2003,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-1.563964,-0.966757
3,wheat,IN,IN-01-0051,2004,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,-0.641504,-1.102767
4,wheat,IN,IN-01-0051,2005,313.989,102.88,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,0.289240,-0.430169


In [22]:
# Final sanity checks

print("panel_z shape:", panel_z.shape)
print("num pair tables:", len(pair_frames))
print("pair names:", sorted(pair_frames.keys()))

panel_z shape: (7319, 46)
num pair tables: 8
pair names: ['avg_prec_yield', 'avg_rsm_avg_fpar', 'avg_rsm_avg_ndvi', 'avg_rsm_yield', 'avg_ssm_avg_fpar', 'avg_ssm_avg_ndvi', 'avg_ssm_yield', 'avg_tavg_avg_cwb']
